## ETL Job One: Parquet file

In [ ]:
from pyspark.sql.functions import col, sum as _sum

# Extract
bookings = spark.table("bookings_csv")
members = spark.table("members_csv")
facilities = spark.table("facilities_csv")

# Transform: total slots booked per facility in September 2012, sorted by slots
# https://pgexercises.com/questions/aggregates/fachoursbymonth.html
fachoursbymonth_df = (
    bookings
    .filter((col("starttime") >= "2012-09-01") & (col("starttime") < "2012-10-01"))
    .groupBy("facid")
    .agg(_sum("slots").alias("Total Slots"))
    .orderBy("Total Slots")
)

display(fachoursbymonth_df)

# Load: write result to a parquet file
parquet_path = "/FileStore/output/fachoursbymonth.parquet"
fachoursbymonth_df.write.mode("overwrite").parquet(parquet_path)

# Sanity check: read it back
display(spark.read.parquet(parquet_path))


## ETL Job Two: Partitions


In [ ]:
from pyspark.sql.functions import col, concat_ws

# Extract
bookings = spark.table("bookings_csv")
members = spark.table("members_csv")
facilities = spark.table("facilities_csv")

# Transform: members who have used a tennis court, with facility name and
# member name as a single column, deduplicated, ordered by member name
# https://pgexercises.com/questions/joins/threejoin.html
threejoin_df = (
    bookings
    .join(members, "memid")
    .join(facilities, "facid")
    .filter(col("name").like("Tennis Court%"))
    .select(
        concat_ws(" ", col("firstname"), col("surname")).alias("member"),
        col("name").alias("facility"),
    )
    .distinct()
    .orderBy("member")
)

display(threejoin_df)

# Load: partition by facility, save as a managed (Delta) table
spark.sql("DROP TABLE IF EXISTS threejoin_delta")

(
    threejoin_df.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("facility")
    .saveAsTable("threejoin_delta")
)

display(spark.sql("SELECT * FROM threejoin_delta"))


## ETL Job Three: HTTP Requests


In [ ]:
import time
import requests
from pyspark.sql import Row
from pyspark.sql.functions import col, date_trunc, max as _max
import dotenv
dotenv.load_dotenv()
# NOTE: You need your own RapidAPI key for the Alpha Vantage API.
api_key = dbutils.secrets.get(scope="alpha-vantage", key="rapidapi-key")

url = "https://alpha-vantage.p.rapidapi.com/query"
headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": api_key,
}

# Google (GOOGL), Apple (AAPL), Microsoft (MSFT), Tesla (TSLA)
companies = ["GOOGL", "AAPL", "MSFT", "TSLA"]

dfs = []

# Extract: loop over each company and fetch its daily time series
for symbol in companies:
    querystring = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "datatype": "json",
        "outputsize": "compact",
    }

    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()

    daily_series = data.get("Time Series (Daily)", {})

    rows = [
        Row(
            company=symbol,
            date=date_str,
            close=float(values["4. close"]),
        )
        for date_str, values in daily_series.items()
    ]

    company_df = spark.createDataFrame(rows)
    dfs.append(company_df)

    # Free tier rate limiting: Alpha Vantage allows 5 requests/minute
    time.sleep(15)

# Union all company DataFrames into a single DF
stocks_df = dfs[0]
for df in dfs[1:]:
    stocks_df = stocks_df.union(df)

# Transform: derive a week column, then find the weekly max closing price per company
weekly_df = stocks_df.withColumn("week", date_trunc("week", col("date").cast("date")))

max_closing_price_weekly_df = (
    weekly_df
    .groupBy("company", "week")
    .agg(_max("close").alias("max_close"))
    .orderBy("company", "week")
)

display(max_closing_price_weekly_df)

# Load: partition by company, save as a managed table
spark.sql("DROP TABLE IF EXISTS max_closing_price_weekly")

(
    max_closing_price_weekly_df.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("company")
    .saveAsTable("max_closing_price_weekly")
)

display(spark.sql("SELECT * FROM max_closing_price_weekly"))


## ETL Job Four: RDBMS

In [ ]:
# Public RNAcentral PostgreSQL database connection details
# https://rnacentral.org/help/public-database
jdbc_hostname = "hh-pgsql-public.ebi.ac.uk"
jdbc_port = 5432
jdbc_database = "pfmegrnargs"
jdbc_url = f"jdbc:postgresql://{jdbc_hostname}:{jdbc_port}/{jdbc_database}"

connection_properties = {
    "user": "reader",
    "password": "NWDMCE5xdipIjRrp",
    "driver": "org.postgresql.Driver",
}

# Extract: read 100 records from the `rna` table via JDBC.
# We push the LIMIT down using a subquery passed as the "table" so only
# 100 rows are pulled across the wire, rather than the whole table.
rna_query = "(SELECT * FROM rna LIMIT 100) AS rna_subset"

rna_df = spark.read.jdbc(
    url=jdbc_url,
    table=rna_query,
    properties=connection_properties,
)

display(rna_df)

# Transform: none required - load the data as-is

# Load
spark.sql("DROP TABLE IF EXISTS rna_100_records")
rna_df.write.saveAsTable("rna_100_records")

display(spark.sql("SELECT * FROM rna_100_records"))
